# 03 · Zero-Shot LLM Evaluation

Evaluates **Claude Haiku 4.5** (`claude-haiku-4-5-20251001`) and
**Claude Sonnet 4.6** (`claude-sonnet-4-6`) on the PhraseBank test set
using zero-shot prompting with structured `<answer>` tag output.

- Temperature 0 for determinism
- 10 concurrent async requests
- Responses cached to `cache/` keyed by text hash — reruns are free
- Exponential-backoff retry on transient errors

**Requires:** `ANTHROPIC_API_KEY` in Colab Secrets or `.env` file.

**Writes:** `results/main_comparison.json` (appends)

In [ ]:
!pip install -q 'datasets>=2.14,<4.0' anthropic python-dotenv tenacity nest_asyncio scikit-learn numpy

## 1 · Configuration & API Key

In [ ]:
# ── Toggle this when running on Colab ──────────────────────────
USE_DRIVE  = False
DRIVE_PATH = '/content/drive/MyDrive/cs443-final-project'
# ───────────────────────────────────────────────────────────────

import asyncio, hashlib, json, os, re, random, time
import numpy as np
from pathlib import Path
import nest_asyncio
nest_asyncio.apply()   # allows asyncio.run() inside Jupyter

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path(DRIVE_PATH)
else:
    BASE = Path('.')

RESULTS_DIR = BASE / 'results'
CACHE_DIR   = BASE / 'cache'
for d in [RESULTS_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Load API key (Colab Secrets → .env fallback)
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    print('API key loaded from Colab Secrets.')
except Exception:
    from dotenv import load_dotenv
    load_dotenv()
    print('API key loaded from .env file.')

import anthropic
client = anthropic.Anthropic()
print('Anthropic client ready.')

## 2 · Model Config & Pricing

In [ ]:
# Model IDs verified against Anthropic docs (April 2025)
MODELS = {
    'haiku':  'claude-haiku-4-5-20251001',
    'sonnet': 'claude-sonnet-4-6',
}

# USD per 1M tokens
PRICING = {
    'claude-haiku-4-5-20251001': {'input': 0.80,  'output': 4.00},
    'claude-sonnet-4-6':         {'input': 3.00,  'output': 15.00},
}

MAX_WORKERS = 10

print('Models configured:')
for short, mid in MODELS.items():
    p = PRICING[mid]
    print(f'  {short}: {mid}  (${p["input"]:.2f}/${ p["output"]:.2f} per 1M tok)')

## 3 · Label Scheme & Metrics Helpers

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

ID2LABEL    = {0: 'negative', 1: 'neutral', 2: 'positive'}
LABEL2ID    = {v: k for k, v in ID2LABEL.items()}
LABEL_NAMES = ['negative', 'neutral', 'positive']


def compute_metrics(y_true, y_pred):
    y_true, y_pred = list(y_true), list(y_pred)
    acc      = float(accuracy_score(y_true, y_pred))
    macro_f1 = float(f1_score(y_true, y_pred, average='macro',
                               labels=[0, 1, 2], zero_division=0))
    per_f1   = f1_score(y_true, y_pred, average=None,
                        labels=[0, 1, 2], zero_division=0)
    cm       = confusion_matrix(y_true, y_pred, labels=[0, 1, 2]).tolist()
    return {
        'accuracy':         acc,
        'macro_f1':         macro_f1,
        'per_class_f1':     {ID2LABEL[i]: float(per_f1[i]) for i in range(3)},
        'confusion_matrix': cm,
    }


def print_metrics(m, title=''):
    if title:
        print('\n' + '─' * 54)
        print('  ' + title)
        print('─' * 54)
    acc = m['accuracy']
    f1  = m['macro_f1']
    print(f'  Accuracy : {acc:.4f}   Macro F1: {f1:.4f}')
    for name, val in m['per_class_f1'].items():
        print(f'  {name:<10}: F1={val:.4f}')

## 4 · Load PhraseBank Test Set

In [ ]:
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split

print('Loading PhraseBank test split...')
raw = load_dataset('gtfintechlab/financial_phrasebank_sentences_allagree')
ds  = raw['train']
if 'sentence' in ds.column_names:
    ds = ds.rename_column('sentence', 'text')

texts  = list(ds['text'])
labels = list(ds['label'])

_, tmp_texts, _, tmp_labels = train_test_split(
    texts, labels, test_size=0.30, random_state=SEED, stratify=labels
)
_, test_texts, _, test_labels = train_test_split(
    tmp_texts, tmp_labels, test_size=0.50, random_state=SEED, stratify=tmp_labels
)

print(f'  Test set: {len(test_texts)} examples')

## 5 · Prompt Template, Cache & Retry

Prompt is intentionally simple and fixed — one version, no chain-of-thought,
no few-shot examples.

In [ ]:
from tenacity import (
    retry, retry_if_exception_type, stop_after_attempt, wait_exponential
)

SYSTEM_PROMPT = 'You are a financial sentiment classifier.'

USER_TEMPLATE = (
    'Classify the following sentence from the perspective of an investor '
    'as positive, negative, or neutral.\n\n'
    'Sentence: {sentence}\n\n'
    'Respond with only one tag: <answer>positive</answer>, '
    '<answer>negative</answer>, or <answer>neutral</answer>.'
)


def cache_key(text):
    return hashlib.sha256(text.encode()).hexdigest()


def load_cache(path):
    cache = {}
    p = Path(path)
    if p.exists():
        with open(p) as f:
            for line in f:
                entry = json.loads(line)
                cache[entry['key']] = entry
    return cache


def append_cache(path, entry):
    with open(path, 'a') as f:
        f.write(json.dumps(entry) + '\n')


def parse_answer(text):
    m = re.search(r'<answer>\s*(positive|negative|neutral)\s*</answer>',
                  text, re.IGNORECASE)
    return m.group(1).lower() if m else None


@retry(
    retry=retry_if_exception_type((
        anthropic.RateLimitError,
        anthropic.InternalServerError,
        anthropic.APIConnectionError,
    )),
    wait=wait_exponential(multiplier=1, min=2, max=60),
    stop=stop_after_attempt(5),
)
def _call_api_sync(model_id, text):
    """Blocking API call with tenacity retry. Wrapped async below."""
    return client.messages.create(
        model=model_id,
        max_tokens=32,
        temperature=0,
        system=SYSTEM_PROMPT,
        messages=[{'role': 'user', 'content': USER_TEMPLATE.format(sentence=text)}],
    )


print('Prompt template and cache helpers ready.')
print('\nSample prompt:')
print(USER_TEMPLATE.format(sentence='The company reported record earnings.'))

## 6 · Async Evaluation Function

Uses `asyncio` with a semaphore to cap concurrency at 10 requests.
Sync API calls are offloaded to a thread pool via `asyncio.to_thread`
so the event loop is never blocked.

In [ ]:
async def evaluate_llm(model_id, texts, true_labels, dataset_name, max_workers=10):
    cache_file = CACHE_DIR / f'llm_{model_id}_{dataset_name}.jsonl'
    cache = load_cache(cache_file)

    predictions   = [None] * len(texts)
    token_in      = 0
    token_out     = 0
    parse_failures = 0
    sem = asyncio.Semaphore(max_workers)

    async def process(i, text):
        nonlocal token_in, token_out, parse_failures
        key = cache_key(text)

        # Cache hit
        if key in cache:
            entry  = cache[key]
            parsed = entry.get('parsed_label')
            if parsed and parsed in LABEL2ID:
                predictions[i] = LABEL2ID[parsed]
                token_in  += entry.get('input_tokens', 0)
                token_out += entry.get('output_tokens', 0)
                return

        # API call (runs in thread pool to not block the event loop)
        async with sem:
            response = await asyncio.to_thread(_call_api_sync, model_id, text)

        response_text = response.content[0].text
        parsed        = parse_answer(response_text)

        entry = {
            'key':           key,
            'text':          text,
            'model':         model_id,
            'dataset':       dataset_name,
            'response_text': response_text,
            'parsed_label':  parsed,
            'input_tokens':  response.usage.input_tokens,
            'output_tokens': response.usage.output_tokens,
        }
        append_cache(cache_file, entry)
        cache[key] = entry

        token_in  += response.usage.input_tokens
        token_out += response.usage.output_tokens

        if parsed and parsed in LABEL2ID:
            predictions[i] = LABEL2ID[parsed]
        else:
            parse_failures += 1
            predictions[i] = LABEL2ID['neutral']   # safe default

    await asyncio.gather(*[process(i, t) for i, t in enumerate(texts)])

    m       = compute_metrics(true_labels, predictions)
    pricing = PRICING.get(model_id, {'input': 0, 'output': 0})
    cost    = (token_in * pricing['input'] + token_out * pricing['output']) / 1_000_000
    m.update({
        'num_examples':        len(texts),
        'total_input_tokens':  token_in,
        'total_output_tokens': token_out,
        'parse_failures':      parse_failures,
        'estimated_cost_usd':  round(cost, 4),
        'cost_per_1k':         round(cost / len(texts) * 1000, 4) if texts else 0,
        'inference_seconds':   0,   # async wall time not tracked per-model
    })
    return m

## 7 · Evaluate Claude Haiku 4.5

In [ ]:
print('=' * 60)
haiku_id = MODELS['haiku']
print(f'Evaluating {haiku_id}')
print(f'Dataset  : PhraseBank test ({len(test_texts)} examples)')
print('=' * 60)

t0 = time.perf_counter()
haiku_metrics = asyncio.run(
    evaluate_llm(haiku_id, test_texts, test_labels, 'phrasebank_test')
)
haiku_metrics['inference_seconds'] = round(time.perf_counter() - t0, 2)

print_metrics(haiku_metrics, 'Claude Haiku 4.5 — PhraseBank Test')
cost = haiku_metrics['estimated_cost_usd']
c1k  = haiku_metrics['cost_per_1k']
print(f'  Cost: ${cost:.4f} total  |  ${c1k:.4f} per 1K predictions')

## 8 · Evaluate Claude Sonnet 4.6

In [ ]:
print('=' * 60)
sonnet_id = MODELS['sonnet']
print(f'Evaluating {sonnet_id}')
print(f'Dataset  : PhraseBank test ({len(test_texts)} examples)')
print('=' * 60)

t0 = time.perf_counter()
sonnet_metrics = asyncio.run(
    evaluate_llm(sonnet_id, test_texts, test_labels, 'phrasebank_test')
)
sonnet_metrics['inference_seconds'] = round(time.perf_counter() - t0, 2)

print_metrics(sonnet_metrics, 'Claude Sonnet 4.6 — PhraseBank Test')
cost = sonnet_metrics['estimated_cost_usd']
c1k  = sonnet_metrics['cost_per_1k']
print(f'  Cost: ${cost:.4f} total  |  ${c1k:.4f} per 1K predictions')

## 9 · Save Results

In [ ]:
results_path = RESULTS_DIR / 'main_comparison.json'
if results_path.exists():
    with open(results_path) as f:
        all_results = json.load(f)
else:
    all_results = {}

all_results['haiku'] = {
    'model':            'haiku',
    'model_id':         haiku_id,
    'phrasebank_test':  haiku_metrics,
}
all_results['sonnet'] = {
    'model':            'sonnet',
    'model_id':         sonnet_id,
    'phrasebank_test':  sonnet_metrics,
}

with open(results_path, 'w') as f:
    json.dump(all_results, f, indent=2)
print('Saved:', results_path)

print('\nSummary:')
for key in ['haiku', 'sonnet']:
    m = all_results[key]['phrasebank_test']
    acc = m['accuracy']
    f1  = m['macro_f1']
    print(f'  {key:<8}  acc={acc:.4f}  macro_f1={f1:.4f}')